# Pipeline Debug Notebook
End-to-end debug of: `train_model` → `evaluate_model` (ingestion) → `score` (scoring)

Run cells top-to-bottom. Each section shows intermediate outputs so you can spot errors at each stage.

In [11]:
import sys
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from xml.etree.ElementTree import parse as parse_xml

# Paths
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'ingestion_program'))
sys.path.insert(0, str(ROOT / 'solution'))
sys.path.insert(0, str(ROOT / 'scoring_program'))

DATA_DIR        = ROOT / 'dev_phase' / 'input_data'
REFERENCE_DIR   = ROOT / 'dev_phase' / 'reference_data'
INGESTION_OUT   = ROOT / 'ingestion_res'
SCORING_OUT     = ROOT / 'scoring_res'

INGESTION_OUT.mkdir(parents=True, exist_ok=True)
SCORING_OUT.mkdir(parents=True, exist_ok=True)

print('ROOT        :', ROOT)
print('DATA_DIR    :', DATA_DIR)
print('REFERENCE   :', REFERENCE_DIR)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.cuda.is_available())

ROOT        : /home/onyxia/work/codalab_tokam2d
DATA_DIR    : /home/onyxia/work/codalab_tokam2d/dev_phase/input_data
REFERENCE   : /home/onyxia/work/codalab_tokam2d/dev_phase/reference_data
PyTorch     : 2.10.0+cu126
CUDA        : True


---
## 1. Train Model

In [13]:
from submission import train_model

training_dir = DATA_DIR / 'train'
print(f'Training directory: {training_dir}')
print(f'Contents: {list(training_dir.iterdir())}')

print('\n--- Starting training ---')
t0 = time.time()
model = train_model(training_dir)
train_time = time.time() - t0

print(f'\n✓ Training complete in {train_time:.1f}s')
print(f'Model type: {type(model)}')

Training directory: /home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train
Contents: [PosixPath('/home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train/turb_i.h5'), PosixPath('/home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train/blob_i.h5'), PosixPath('/home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train/blob_dwi.xml'), PosixPath('/home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train/blob_i.xml'), PosixPath('/home/onyxia/work/codalab_tokam2d/dev_phase/input_data/train/blob_dwi.h5')]

--- Starting training ---
Current working directory: /home/onyxia/work/codalab_tokam2d
Dataset directory: /home/onyxia/work/codalab_tokam2d/yolo_training_data
Loading dataset...
Found annotations for 30 frames in 2 files.
Loaded 30 frames.
Total labeled samples: 30
Training: 30 samples (using all labeled data)
Validation: 1 samples (minimal set for YOLO)

Dataset verification:
  Train images: 30
  Val images: 1 (minimal set)
Dataset preparation complete!
Config file: /hom

---
## 2. Run Inference on a Few Frames
Quick sanity check — run the model on individual images and visualise the predictions.

In [14]:
from tokam2d_utils import TokamDataset

# Load a small slice of the training set so we can inspect predictions
train_dataset = TokamDataset(training_dir, include_unlabeled=False)
print(f'Dataset size: {len(train_dataset)}')

model.eval()

NUM_SAMPLES = 6
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for plot_idx in range(NUM_SAMPLES):
    img_tensor, target = train_dataset[plot_idx]

    with torch.no_grad():
        preds = model((img_tensor,))  # model expects a tuple
    pred = preds[0]

    # Display image
    img_np = img_tensor.squeeze().numpy()
    img_norm = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    axes[plot_idx].imshow(img_norm, cmap='gray')
    axes[plot_idx].axis('off')

    # Ground truth (green)
    if target['boxes'] is not None and len(target['boxes']) > 0:
        for box in target['boxes']:
            b = box.numpy() if hasattr(box, 'numpy') else np.array(box)
            h, w = img_np.shape
            if b[0] > 1:  # pixel coords
                x1, y1, bw, bh = b[0], b[1], b[2], b[3]
            else:          # normalised
                x1, y1, bw, bh = b[0]*w, b[1]*h, b[2]*w, b[3]*h
            rect = patches.Rectangle((x1, y1), bw, bh,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            axes[plot_idx].add_patch(rect)

    # Predictions (red)
    det_count = 0
    if pred['boxes'] is not None and len(pred['boxes']) > 0:
        for box, score in zip(pred['boxes'], pred['scores']):
            x1, y1, x2, y2 = box.numpy()
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='red',
                                      linestyle='--', facecolor='none')
            axes[plot_idx].add_patch(rect)
            axes[plot_idx].text(x1, y1-4, f'{score:.2f}', color='red',
                                fontsize=9, weight='bold',
                                bbox=dict(facecolor='black', alpha=0.5, pad=1))
            det_count += 1

    gt_count = len(target['boxes']) if target['boxes'] is not None else 0
    fi = target.get('frame_index', plot_idx)
    axes[plot_idx].set_title(f'{fi} | GT:{gt_count}  Pred:{det_count}', fontsize=10)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor='none', edgecolor='lime', linewidth=2, label='Ground Truth'),
    Patch(facecolor='none', edgecolor='red',  linewidth=2, linestyle='--', label='Prediction')
], loc='upper center', ncol=2, fontsize=12)
plt.suptitle('Direct Model Inference (Training Frames)', fontsize=14, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

Found annotations for 30 frames in 2 files.
Loaded 30 frames.
Dataset size: 30
  Detected 55 blobs, scores: [0.0429086871445179, 0.03309626132249832, 0.03262748569250107, 0.03163375332951546, 0.02656998112797737, 0.024826155975461006, 0.023002559319138527, 0.022404439747333527, 0.02173766866326332, 0.021202389150857925, 0.0202312134206295, 0.01975840888917446, 0.019558925181627274, 0.018664320930838585, 0.01859201490879059, 0.018094228580594063, 0.0180673785507679, 0.01799921505153179, 0.017986876890063286, 0.017775485292077065, 0.017492929473519325, 0.016709983348846436, 0.016543693840503693, 0.016031118109822273, 0.01599060371518135, 0.015804700553417206, 0.01572025753557682, 0.014963650144636631, 0.014857450500130653, 0.014601228758692741, 0.014597668312489986, 0.014018112793564796, 0.014002744108438492, 0.013737836852669716, 0.01311150100082159, 0.013063195161521435, 0.013004034757614136, 0.012939205393195152, 0.012842626310884953, 0.012439867481589317, 0.01243764627724886, 0.01231

<Figure size 1800x1200 with 6 Axes>

---
## 3. Run Ingestion (evaluate_model)
Reproduces `ingestion.py` locally and writes prediction XMLs.

In [15]:

from tokam2d_utils.xml_loader import dump_to_xml

# Score against training data (we don't have test ground truth)
EVAL_SETS = ['train']

def collate_fn(batch):
    return tuple(zip(*batch))

def evaluate_model(model, data_dir):
    eval_dataset = TokamDataset(data_dir)
    eval_dataloader = torch.utils.data.DataLoader(
        eval_dataset, batch_size=2, collate_fn=collate_fn
    )
    model.eval()
    res = []
    for X, y in eval_dataloader:
        with torch.no_grad():
            y_pred = model(X)
        y_pred = [
            {**y_p, 'frame_index': y_t['frame_index']}
            for y_p, y_t in zip(y_pred, y)
        ]
        res.extend(y_pred)
    return res

all_results = {}
t0 = time.time()
for eval_set in EVAL_SETS:
    eval_dir = DATA_DIR / eval_set
    if not eval_dir.exists():
        print(f'⚠ Skipping {eval_set} — directory not found: {eval_dir}')
        continue
    print(f'Evaluating {eval_set} ...')
    results = evaluate_model(model, eval_dir)
    all_results[eval_set] = results
    out_xml = INGESTION_OUT / f'{eval_set}_predictions.xml'
    dump_to_xml(results, out_xml)
    print(f'  → {len(results)} frames written to {out_xml}')
test_time = time.time() - t0

# Write metadata
meta = dict(train_time=train_time, test_time=test_time)
with open(INGESTION_OUT / 'metadata.json', 'w') as f:
    json.dump(meta, f)

print(f'\n✓ Ingestion done in {test_time:.1f}s')
print(f'metadata: {meta}')


Evaluating train ...
Found annotations for 30 frames in 2 files.
Loaded 30 frames.
  Detected 55 blobs, scores: [0.0429086871445179, 0.03309626132249832, 0.03262748569250107, 0.03163375332951546, 0.02656998112797737, 0.024826155975461006, 0.023002559319138527, 0.022404439747333527, 0.02173766866326332, 0.021202389150857925, 0.0202312134206295, 0.01975840888917446, 0.019558925181627274, 0.018664320930838585, 0.01859201490879059, 0.018094228580594063, 0.0180673785507679, 0.01799921505153179, 0.017986876890063286, 0.017775485292077065, 0.017492929473519325, 0.016709983348846436, 0.016543693840503693, 0.016031118109822273, 0.01599060371518135, 0.015804700553417206, 0.01572025753557682, 0.014963650144636631, 0.014857450500130653, 0.014601228758692741, 0.014597668312489986, 0.014018112793564796, 0.014002744108438492, 0.013737836852669716, 0.01311150100082159, 0.013063195161521435, 0.013004034757614136, 0.012939205393195152, 0.012842626310884953, 0.012439867481589317, 0.01243764627724886, 0.0

---
## 4. Inspect Prediction XML
Parse and show what was written to the XML file.

In [16]:
for eval_set in EVAL_SETS:
    xml_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if not xml_path.exists():
        print(f'⚠ {xml_path} not found')
        continue

    tree = parse_xml(xml_path)
    root = tree.getroot()
    images = root.findall('image')

    total_boxes = sum(len(img.findall('box')) for img in images)
    frames_with_det = sum(1 for img in images if len(img.findall('box')) > 0)

    print(f'\n=== {eval_set}_predictions.xml ===')
    print(f'  Total frames  : {len(images)}')
    print(f'  Frames w/ detections: {frames_with_det}')
    print(f'  Total boxes   : {total_boxes}')

    if total_boxes == 0:
        print('  ⚠ WARNING: No detections at all — model may not have converged')

    print('\n  First 10 frames:')
    for img in images[:10]:
        boxes = img.findall('box')
        scores = [float(b.attrib.get('score', 0)) for b in boxes]
        score_str = f'scores={[f"{s:.3f}" for s in scores]}' if scores else 'no detections'
        print(f'    {img.attrib["index"]:25s}  {len(boxes)} boxes  {score_str}')


=== train_predictions.xml ===
  Total frames  : 30
  Frames w/ detections: 30
  Total boxes   : 7568

  First 10 frames:
    blob_i-50                  55 boxes  scores=['0.043', '0.033', '0.033', '0.032', '0.027', '0.025', '0.023', '0.022', '0.022', '0.021', '0.020', '0.020', '0.020', '0.019', '0.019', '0.018', '0.018', '0.018', '0.018', '0.018', '0.017', '0.017', '0.017', '0.016', '0.016', '0.016', '0.016', '0.015', '0.015', '0.015', '0.015', '0.014', '0.014', '0.014', '0.013', '0.013', '0.013', '0.013', '0.013', '0.012', '0.012', '0.012', '0.012', '0.012', '0.012', '0.011', '0.011', '0.011', '0.011', '0.011', '0.011', '0.010', '0.010', '0.010', '0.010']
    blob_i-100                 71 boxes  scores=['0.143', '0.057', '0.050', '0.042', '0.041', '0.039', '0.038', '0.030', '0.029', '0.028', '0.026', '0.026', '0.026', '0.024', '0.024', '0.023', '0.023', '0.023', '0.022', '0.022', '0.022', '0.021', '0.021', '0.021', '0.021', '0.020', '0.020', '0.020', '0.020', '0.019', '0.019', '0.019

---
## 5. Compare Predictions vs Ground Truth (Visual)
Side-by-side visualisation on test frames.

In [17]:
eval_set = 'test'
pred_xml_path = INGESTION_OUT / f'{eval_set}_predictions.xml'

# Load predictions from XML
def read_xml_preds(path):
    tree = parse_xml(path)
    root = tree.getroot()
    out = {}
    for img in root.findall('image'):
        idx = img.attrib.get('index', img.attrib.get('name', '?'))
        boxes = []
        scores = []
        for b in img.findall('box'):
            boxes.append((
                float(b.attrib['xtl']), float(b.attrib['ytl']),
                float(b.attrib['xbr']), float(b.attrib['ybr'])
            ))
            scores.append(float(b.attrib.get('score', 1.0)))
        out[idx] = {'boxes': boxes, 'scores': scores}
    return out

if not pred_xml_path.exists():
    print(f'⚠ {pred_xml_path} not found — run ingestion first (cell 4)')
else:
    predictions = read_xml_preds(pred_xml_path)

    # Load test frames
    test_dir = DATA_DIR / eval_set
    test_dataset = TokamDataset(test_dir, include_unlabeled=True)
    NUM_SHOW = min(6, len(test_dataset))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for plot_idx in range(NUM_SHOW):
        img_tensor, target = test_dataset[plot_idx]
        frame_idx = str(target['frame_index'])

        img_np = img_tensor.squeeze().numpy()
        img_norm = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        axes[plot_idx].imshow(img_norm, cmap='gray')
        axes[plot_idx].axis('off')

        # Predictions from XML
        frame_key = frame_idx.split('-')[-1] if '-' in frame_idx else frame_idx
        full_key = next((k for k in predictions if k == frame_idx), None) or \
                   next((k for k in predictions if frame_key in k), None)

        pred_count = 0
        if full_key and predictions[full_key]['boxes']:
            for (x1, y1, x2, y2), score in zip(
                predictions[full_key]['boxes'],
                predictions[full_key]['scores']
            ):
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor='red',
                                          linestyle='--', facecolor='none')
                axes[plot_idx].add_patch(rect)
                axes[plot_idx].text(x1, y1-4, f'{score:.2f}',
                                    color='red', fontsize=9, weight='bold',
                                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
                pred_count += 1

        axes[plot_idx].set_title(f'{frame_idx} | Pred:{pred_count}', fontsize=10)

    plt.suptitle('Predictions on Test Frames (from XML)', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Total prediction keys in XML: {len(predictions)}')

No label files found
Loaded 4 frames.


<Figure size 1800x1200 with 6 Axes>

Total prediction keys in XML: 4


---
## 6. Run Scoring (against Training XMLs)
Computes AP50 using IoMean. Since we don't have test ground truth, we score against
the training annotations (`blob_i.xml`, `blob_dwi.xml`).
Frame keys are prefixed with the simulation name (e.g. `blob_i-50`) to avoid collisions
between simulations that share the same frame numbers.


In [18]:

from scoring import compute_ap
from xml.etree.ElementTree import parse as parse_xml_et

def read_training_xml(xml_path, sim_name):
    """Read a training annotation XML. Returns dict keyed by '{sim_name}-{frame_num}'
    to match the prediction index format (e.g. 'blob_i-50') and avoid collisions
    between simulations that share the same frame numbers."""
    tree = parse_xml_et(xml_path)
    root = tree.getroot()
    annotations = {}
    for element in root.findall("image"):
        if "name" in element.attrib:
            frame_num = element.attrib["name"].split(".")[0]
        else:
            frame_num = element.attrib["index"].split("-")[-1]
        key = f"{sim_name}-{frame_num}"
        annotations[key] = {
            "boxes": [
                (
                    float(b.attrib["xtl"]), float(b.attrib["ytl"]),
                    float(b.attrib["xbr"]), float(b.attrib["ybr"]),
                )
                for b in element.findall("box")
            ],
            "scores": [float(b.attrib.get("score", 1.0)) for b in element.findall("box")],
        }
    return annotations

def read_pred_xml_full(xml_path):
    """Read predictions XML keeping the full index (e.g. 'blob_i-50')."""
    tree = parse_xml_et(xml_path)
    root = tree.getroot()
    annotations = {}
    for element in root.findall("image"):
        key = element.attrib.get("index", element.attrib.get("name", "?"))
        annotations[key] = {
            "boxes": [
                (
                    float(b.attrib["xtl"]), float(b.attrib["ytl"]),
                    float(b.attrib["xbr"]), float(b.attrib["ybr"]),
                )
                for b in element.findall("box")
            ],
            "scores": [float(b.attrib.get("score", 1.0)) for b in element.findall("box")],
        }
    return annotations

# Collect all training XML files and build a merged reference dict
train_dir = DATA_DIR / 'train'
training_xmls = sorted(train_dir.glob('*.xml'))
if not training_xmls:
    print(f'⚠ No XML files found in {train_dir}')
else:
    print(f'Found {len(training_xmls)} training XML files:')
    for xml_path in training_xmls:
        print(f'  {xml_path.name}')

    # Build merged reference keyed by sim-frame (e.g. 'blob_i-50')
    targets_all = {}
    for xml_path in training_xmls:
        sim_name = xml_path.stem  # e.g. 'blob_i'
        sim_targets = read_training_xml(xml_path, sim_name)
        targets_all.update(sim_targets)

    print(f'\nTotal reference frames (labeled): {len(targets_all)}')
    total_gt_boxes = sum(len(v["boxes"]) for v in targets_all.values())
    print(f'Total GT boxes: {total_gt_boxes}')

    # Load predictions (full index, not split)
    pred_path = INGESTION_OUT / 'train_predictions.xml'
    if not pred_path.exists():
        print(f'\n⚠ {pred_path} not found — run ingestion (cell 3) first')
    else:
        predictions_all = read_pred_xml_full(pred_path)
        print(f'\nPrediction frames: {len(predictions_all)}')
        total_pred_boxes = sum(len(v["boxes"]) for v in predictions_all.values())
        frames_w_det = sum(1 for v in predictions_all.values() if v["boxes"])
        print(f'Total pred boxes: {total_pred_boxes}  |  Frames with detections: {frames_w_det}')

        matched = set(predictions_all.keys()) & set(targets_all.keys())
        print(f'Matched frames (pred ∩ GT):  {len(matched)}')
        unmatched_pred = set(predictions_all.keys()) - set(targets_all.keys())
        if unmatched_pred:
            print(f'  ⚠ Pred frames not in GT ({len(unmatched_pred)}): {list(unmatched_pred)[:5]}')

        ap50 = compute_ap(predictions_all, targets_all, threshold=0.5)
        print(f'\n  ★  AP50 (train set): {ap50:.4f}')

        scores = {'train_ap50': ap50}
        meta = json.loads((INGESTION_OUT / 'metadata.json').read_text())
        scores.update(**meta)

        SCORING_OUT.mkdir(parents=True, exist_ok=True)
        (SCORING_OUT / 'scores.json').write_text(json.dumps(scores, indent=2))
        print(f'\n✓ Written to {SCORING_OUT / "scores.json"}')
        print(json.dumps(scores, indent=2))


Found 2 training XML files:
  blob_dwi.xml
  blob_i.xml

Total reference frames (labeled): 30
Total GT boxes: 282

Prediction frames: 30
Total pred boxes: 7568  |  Frames with detections: 30
Matched frames (pred ∩ GT):  30

  ★  AP50 (train set): 0.0000

✓ Written to /home/onyxia/work/codalab_tokam2d/scoring_res/scores.json
{
  "train_ap50": 0.0,
  "train_time": 92.80025935173035,
  "test_time": 4.13235878944397
}


---
## 7. Score Breakdown per Frame
Shows per-frame IoMean between predictions and ground truth.

In [19]:

from discopat.metrics import compute_iomean

pred_path = INGESTION_OUT / 'train_predictions.xml'
if not pred_path.exists():
    print(f'⚠ Run ingestion (cell 3) first.')
elif not training_xmls:
    print(f'⚠ No training XMLs found (run scoring cell first).')
else:
    # Reuse targets_all and predictions_all from scoring cell above
    frame_results = []
    for frame_key, target in targets_all.items():
        gt_boxes = target['boxes']
        if frame_key not in predictions_all or not predictions_all[frame_key]['boxes']:
            frame_results.append({'frame': frame_key, 'gt': len(gt_boxes),
                                   'pred': 0, 'best_iomean': 0.0})
            continue
        pred_boxes = predictions_all[frame_key]['boxes']
        best = max(
            (compute_iomean(p, g) for p in pred_boxes for g in gt_boxes),
            default=0.0
        )
        frame_results.append({'frame': frame_key, 'gt': len(gt_boxes),
                               'pred': len(pred_boxes), 'best_iomean': best})

    frame_results.sort(key=lambda x: x['best_iomean'])

    print(f'Per-frame results (train):'
          f'\n{"Frame":30s} {"GT":>4} {"Pred":>5} {"Best IoMean":>12}')
    print('-' * 55)
    for r in frame_results:
        tp_flag = '✓' if r['best_iomean'] >= 0.5 else '✗'
        print(f"{r['frame']:30s} {r['gt']:>4} {r['pred']:>5} "
              f"{r['best_iomean']:>12.3f}  {tp_flag}")

    tp_count = sum(1 for r in frame_results if r['best_iomean'] >= 0.5)
    print(f'\nFrames with IoMean >= 0.5: {tp_count}/{len(frame_results)}')

    # Plot IoMean distribution
    iomeans = [r['best_iomean'] for r in frame_results]
    plt.figure(figsize=(10, 4))
    plt.hist(iomeans, bins=20, color='steelblue', edgecolor='white')
    plt.axvline(0.5, color='red', linestyle='--', label='threshold=0.5')
    plt.xlabel('Best IoMean per frame')
    plt.ylabel('Count')
    plt.title('Per-frame IoMean distribution (train set)')
    plt.legend()
    plt.tight_layout()
    plt.show()


Per-frame results (train):
Frame                            GT  Pred  Best IoMean
-------------------------------------------------------
blob_i-200                        2   158        0.081  ✗
blob_dwi-150                      2   189        0.097  ✗
blob_dwi-250                      5   261        0.124  ✗
blob_i-150                        1   118        0.136  ✗
blob_i-300                        6   224        0.137  ✗
blob_dwi-550                     18   300        0.152  ✗
blob_i-100                        1    71        0.154  ✗
blob_dwi-50                       1    69        0.203  ✗
blob_dwi-650                     17   300        0.226  ✗
blob_dwi-450                     13   300        0.229  ✗
blob_dwi-350                     10   294        0.239  ✗
blob_i-850                       12   300        0.243  ✗
blob_i-400                       10   273        0.250  ✗
blob_i-750                       10   300        0.264  ✗
blob_i-950                       12   300        0

<Figure size 1000x400 with 1 Axes>

---
## 8. Confidence Score Distribution
Histogram of all prediction confidence scores.

In [20]:
for eval_set in EVAL_SETS:
    pred_path = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if not pred_path.exists():
        continue

    tree = parse_xml(pred_path)
    root = tree.getroot()
    all_scores = [
        float(b.attrib.get('score', 1.0))
        for img in root.findall('image')
        for b in img.findall('box')
    ]

    if not all_scores:
        print(f'{eval_set}: No detections to show')
        continue

    print(f'{eval_set}: {len(all_scores)} total detections')
    print(f'  min={min(all_scores):.3f}  max={max(all_scores):.3f}  '
          f'mean={np.mean(all_scores):.3f}  median={np.median(all_scores):.3f}')

    plt.figure(figsize=(10, 4))
    plt.hist(all_scores, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(0.25, color='orange', linestyle='--', label='default YOLO conf=0.25')
    plt.axvline(0.5,  color='red',    linestyle='--', label='conf=0.5')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title(f'Confidence Distribution — {eval_set}')
    plt.legend()
    plt.tight_layout()
    plt.show()

train: 7568 total detections
  min=0.010  max=0.197  mean=0.027  median=0.021


<Figure size 1000x400 with 1 Axes>

---
## 9. Checklist Summary

In [21]:

print('=' * 60)
print('PIPELINE CHECKLIST')
print('=' * 60)

train_xml_exists = any((DATA_DIR / 'train').glob('*.xml'))
pred_xml_exists  = (INGESTION_OUT / 'train_predictions.xml').exists()

checks = [
    ('Model trained',                model is not None),
    ('Training XMLs found',          train_xml_exists),
    ('train evaluation run',         pred_xml_exists),
    ('metadata.json written',        (INGESTION_OUT / 'metadata.json').exists()),
    ('scores.json written',          (SCORING_OUT / 'scores.json').exists()),
]

for label, ok in checks:
    status = '✓' if ok else '✗'
    print(f'  {status}  {label}')

if (SCORING_OUT / 'scores.json').exists():
    final = json.loads((SCORING_OUT / 'scores.json').read_text())
    print(f'\nFinal scores:')
    for k, v in final.items():
        print(f'  {k:25s}: {v}')

# Detections summary
print()
for eval_set in EVAL_SETS:
    p = INGESTION_OUT / f'{eval_set}_predictions.xml'
    if p.exists():
        tree = parse_xml(p)
        total_boxes = sum(len(img.findall('box')) for img in tree.getroot().findall('image'))
        frames = len(tree.getroot().findall('image'))
        print(f'  {eval_set}: {total_boxes} total detections across {frames} frames')
        if total_boxes == 0:
            print(f'  ⚠ No detections — check training convergence or conf threshold')

print('\nNote: Scoring is done on TRAINING data (no test ground truth available).')
print('To get official scores, submit to the Codabench challenge.')


PIPELINE CHECKLIST
  ✓  Model trained
  ✓  Training XMLs found
  ✓  train evaluation run
  ✓  metadata.json written
  ✓  scores.json written

Final scores:
  train_ap50               : 0.0
  train_time               : 92.80025935173035
  test_time                : 4.13235878944397

  train: 7568 total detections across 30 frames

Note: Scoring is done on TRAINING data (no test ground truth available).
To get official scores, submit to the Codabench challenge.
